### Merged pdfs with bookmark structure.

Perplexity discourse [here](https://www.perplexity.ai/search/fix-the-bug-in-the-code-below-plw_2PR4TUWH6xqSZ7M_nQ#20).

In [9]:
from pypdf import PdfWriter, PdfReader
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from datetime import datetime
import io
import os

def create_first_page(writer, pdf_files):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=letter)
    
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    c.setFont("Helvetica-Bold", 14)
    header = f"{len(pdf_files)} PDFs merged on {timestamp}"
    c.drawString(72, 750, header)
    
    c.line(72, 735, 540, 735)
    
    c.setFont("Helvetica", 12)
    y_position = 700
    
    for i, pdf_path in enumerate(pdf_files, 1):
        pdf_basename = os.path.basename(pdf_path).split('.')[0]
        title = f"Article {i}: {pdf_basename}"
        c.drawString(72, y_position, title)
        y_position -= 20
        
        if y_position < 50:
            c.showPage()
            c.setFont("Helvetica", 12)
            y_position = 750
    
    c.showPage()
    c.save()
    packet.seek(0)
    first_pdf = PdfReader(packet)
    writer.add_page(first_pdf.pages[0])

def create_sample_pdf(filename, title, bookmarks, margin_text):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=letter)
    
    # Add vertical text in right margin - now larger and closer to edge
    c.saveState()
    c.setFont("Helvetica-Bold", 14)  # Increased from 10 to 14
    c.setFillColor(colors.darkgreen)
    c.translate(580, 400)  # Moved from 550 to 580
    c.rotate(90)
    c.drawString(0, 0, margin_text)
    c.restoreState()
    
    # Main content
    c.setFont("Helvetica", 12)
    c.setFillColor(colors.black)
    c.drawString(100, 750, title)
    c.showPage()
    c.save()
    packet.seek(0)

    new_pdf = PdfReader(packet)
    writer = PdfWriter()
    writer.add_page(new_pdf.pages[0])

    for bookmark_title, page in bookmarks:
        writer.add_outline_item(bookmark_title, page)

    with open(filename, 'wb') as f:
        writer.write(f)

def add_separator_page(writer, pdf_basename, full_path, metadata):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=letter)
    
    c.setFont("Helvetica-Bold", 16)
    c.setFillColorRGB(0, 0, 1)  # Blue
    
    text = pdf_basename
    x, y = 100, 600
    c.linkURL(full_path, (x, y-5, x+200, y+20), relative=1)
    c.drawString(x, y, text)
    
    c.setFont("Helvetica", 12)
    c.setFillColorRGB(0, 0, 0)  # Black
    y_position = 500
    for key, value in metadata.items():
        c.drawString(100, y_position, f"{key}: {value}")
        y_position -= 30
    
    c.showPage()
    c.save()
    packet.seek(0)
    separator_pdf = PdfReader(packet)
    writer.add_page(separator_pdf.pages[0])

def merge_pdfs_with_structure(pdf_files, output_path):
    writer = PdfWriter()
    
    sample_metadata = [
        {
            "Title": "First Research Paper",
            "Author": "John Smith",
            "Source": "Science Journal",
            "Date": "2024-01-15"
        },
        {
            "Title": "Second Research Paper",
            "Author": "Jane Doe",
            "Source": "Nature",
            "Date": "2024-02-20"
        },
        {
            "Title": "Third Research Paper",
            "Author": "Bob Johnson",
            "Source": "Research Quarterly",
            "Date": "2024-03-10"
        }
    ]
    
    create_first_page(writer, pdf_files)
    current_page = 1
    
    for i, pdf_path in enumerate(pdf_files, 1):
        pdf_basename = os.path.basename(pdf_path).split('.')[0]
        full_path = os.path.abspath(pdf_path)
        
        add_separator_page(writer, pdf_basename, full_path, sample_metadata[i-1])
        article_title = f"Article {i}: {pdf_basename}"
        separator_bookmark = writer.add_outline_item(article_title, current_page)
        current_page += 1
        
        pdf = PdfReader(pdf_path)
        page_offset = current_page
        
        for page in pdf.pages:
            writer.add_page(page)
            
        if pdf.outline:
            seen_bookmarks = set()
            for item in pdf.outline:
                if isinstance(item, dict) and '/Page' in item:
                    title = item['/Title']
                    if title not in seen_bookmarks:
                        seen_bookmarks.add(title)
                        page_num = pdf.get_destination_page_number(item)
                        writer.add_outline_item(
                            title,
                            page_offset + page_num,
                            parent=separator_bookmark
                        )
        
        current_page += len(pdf.pages)
    
    with open(output_path, 'wb') as output:
        writer.write(output)

def verify_bookmarks(pdf_path):
    reader = PdfReader(pdf_path)
    
    def print_bookmark_tree(bookmarks, level=0):
        for item in bookmarks:
            if isinstance(item, list):
                print_bookmark_tree(item, level + 1)
            else:
                indent = "  " * level
                page_num = reader.get_destination_page_number(item)
                print(f"{indent}- {item.title} (Page {page_num})")
    
    print("\nBookmark structure:")
    print_bookmark_tree(reader.outline)

def run_tests():
    print("Creating sample PDFs...")
    
    create_sample_pdf("article1.pdf", "Article 1", [
        ("Section 1.1", 0),
        ("Section 1.2", 0)
    ], "article1")
    
    create_sample_pdf("article2.pdf", "Article 2", [
        ("Section 2.1", 0),
        ("Section 2.2", 0)
    ], "article2")
    
    create_sample_pdf("article3.pdf", "Article 3", [
        ("Section 3.1", 0),
        ("Section 3.2", 0)
    ], "article3")
    
    print("Merging PDFs...")
    pdf_files = ["article1.pdf", "article2.pdf", "article3.pdf"]
    merge_pdfs_with_structure(pdf_files, "merged_articles.pdf")
    
    print("Verifying merged PDF structure...")
    verify_bookmarks("merged_articles.pdf")
    
    print("\nTest completed. Please check merged_articles.pdf")

if __name__ == "__main__":
    run_tests()


Creating sample PDFs...
Merging PDFs...
Verifying merged PDF structure...

Bookmark structure:
- Article 1: article1 (Page 1)
  - Section 1.1 (Page 2)
  - Section 1.2 (Page 2)
- Article 2: article2 (Page 3)
  - Section 2.1 (Page 4)
  - Section 2.2 (Page 4)
- Article 3: article3 (Page 5)
  - Section 3.1 (Page 6)
  - Section 3.2 (Page 6)

Test completed. Please check merged_articles.pdf


In [10]:
# Has almost everything, but margin text too small and far far from the RHS of the page.

# from pypdf import PdfWriter, PdfReader
# from reportlab.pdfgen import canvas
# from reportlab.lib.pagesizes import letter
# from reportlab.lib import colors
# from datetime import datetime
# import io
# import os

# def create_first_page(writer, pdf_files):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
    
#     timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
#     c.setFont("Helvetica-Bold", 14)
#     header = f"{len(pdf_files)} PDFs merged on {timestamp}"
#     c.drawString(72, 750, header)
    
#     c.line(72, 735, 540, 735)
    
#     c.setFont("Helvetica", 12)
#     y_position = 700
    
#     for i, pdf_path in enumerate(pdf_files, 1):
#         pdf_basename = os.path.basename(pdf_path).split('.')[0]
#         title = f"Article {i}: {pdf_basename}"
#         c.drawString(72, y_position, title)
#         y_position -= 20
        
#         if y_position < 50:
#             c.showPage()
#             c.setFont("Helvetica", 12)
#             y_position = 750
    
#     c.showPage()
#     c.save()
#     packet.seek(0)
#     first_pdf = PdfReader(packet)
#     writer.add_page(first_pdf.pages[0])

# def create_sample_pdf(filename, title, bookmarks, margin_text):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
    
#     # Add vertical text in right margin
#     c.saveState()
#     c.setFont("Helvetica-Bold", 10)
#     c.setFillColor(colors.darkgreen)
#     c.translate(550, 400)  # Position in right margin
#     c.rotate(90)
#     c.drawString(0, 0, margin_text)
#     c.restoreState()
    
#     # Main content
#     c.setFont("Helvetica", 12)
#     c.setFillColor(colors.black)
#     c.drawString(100, 750, title)
#     c.showPage()
#     c.save()
#     packet.seek(0)

#     new_pdf = PdfReader(packet)
#     writer = PdfWriter()
#     writer.add_page(new_pdf.pages[0])

#     for bookmark_title, page in bookmarks:
#         writer.add_outline_item(bookmark_title, page)

#     with open(filename, 'wb') as f:
#         writer.write(f)

# def add_separator_page(writer, pdf_basename, full_path, metadata):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
    
#     c.setFont("Helvetica-Bold", 16)
#     c.setFillColorRGB(0, 0, 1)  # Blue
    
#     text = pdf_basename
#     x, y = 100, 600
#     c.linkURL(full_path, (x, y-5, x+200, y+20), relative=1)
#     c.drawString(x, y, text)
    
#     c.setFont("Helvetica", 12)
#     c.setFillColorRGB(0, 0, 0)  # Black
#     y_position = 500
#     for key, value in metadata.items():
#         c.drawString(100, y_position, f"{key}: {value}")
#         y_position -= 30
    
#     c.showPage()
#     c.save()
#     packet.seek(0)
#     separator_pdf = PdfReader(packet)
#     writer.add_page(separator_pdf.pages[0])

# def merge_pdfs_with_structure(pdf_files, output_path):
#     writer = PdfWriter()
    
#     sample_metadata = [
#         {
#             "Title": "First Research Paper",
#             "Author": "John Smith",
#             "Source": "Science Journal",
#             "Date": "2024-01-15"
#         },
#         {
#             "Title": "Second Research Paper",
#             "Author": "Jane Doe",
#             "Source": "Nature",
#             "Date": "2024-02-20"
#         },
#         {
#             "Title": "Third Research Paper",
#             "Author": "Bob Johnson",
#             "Source": "Research Quarterly",
#             "Date": "2024-03-10"
#         }
#     ]
    
#     create_first_page(writer, pdf_files)
#     current_page = 1
    
#     for i, pdf_path in enumerate(pdf_files, 1):
#         pdf_basename = os.path.basename(pdf_path).split('.')[0]
#         full_path = os.path.abspath(pdf_path)
        
#         add_separator_page(writer, pdf_basename, full_path, sample_metadata[i-1])
#         article_title = f"Article {i}: {pdf_basename}"
#         separator_bookmark = writer.add_outline_item(article_title, current_page)
#         current_page += 1
        
#         pdf = PdfReader(pdf_path)
#         page_offset = current_page
        
#         for page in pdf.pages:
#             writer.add_page(page)
            
#         if pdf.outline:
#             seen_bookmarks = set()
#             for item in pdf.outline:
#                 if isinstance(item, dict) and '/Page' in item:
#                     title = item['/Title']
#                     if title not in seen_bookmarks:
#                         seen_bookmarks.add(title)
#                         page_num = pdf.get_destination_page_number(item)
#                         writer.add_outline_item(
#                             title,
#                             page_offset + page_num,
#                             parent=separator_bookmark
#                         )
        
#         current_page += len(pdf.pages)
    
#     with open(output_path, 'wb') as output:
#         writer.write(output)

# def verify_bookmarks(pdf_path):
#     reader = PdfReader(pdf_path)
    
#     def print_bookmark_tree(bookmarks, level=0):
#         for item in bookmarks:
#             if isinstance(item, list):
#                 print_bookmark_tree(item, level + 1)
#             else:
#                 indent = "  " * level
#                 page_num = reader.get_destination_page_number(item)
#                 print(f"{indent}- {item.title} (Page {page_num})")
    
#     print("\nBookmark structure:")
#     print_bookmark_tree(reader.outline)

# def run_tests():
#     print("Creating sample PDFs...")
    
#     create_sample_pdf("article1.pdf", "Article 1", [
#         ("Section 1.1", 0),
#         ("Section 1.2", 0)
#     ], "article1")
    
#     create_sample_pdf("article2.pdf", "Article 2", [
#         ("Section 2.1", 0),
#         ("Section 2.2", 0)
#     ], "article2")
    
#     create_sample_pdf("article3.pdf", "Article 3", [
#         ("Section 3.1", 0),
#         ("Section 3.2", 0)
#     ], "article3")
    
#     print("Merging PDFs...")
#     pdf_files = ["article1.pdf", "article2.pdf", "article3.pdf"]
#     merge_pdfs_with_structure(pdf_files, "merged_articles.pdf")
    
#     print("Verifying merged PDF structure...")
#     verify_bookmarks("merged_articles.pdf")
    
#     print("\nTest completed. Please check merged_articles.pdf")

# if __name__ == "__main__":
#     run_tests()


In [11]:
# Almost perfect.  
#
# from pypdf import PdfWriter, PdfReader
# from reportlab.pdfgen import canvas
# from reportlab.lib.pagesizes import letter
# from reportlab.lib import colors
# from datetime import datetime
# import io
# import os

# def create_first_page(writer, pdf_files):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
    
#     # Add timestamp and PDF count header
#     timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
#     c.setFont("Helvetica-Bold", 14)
#     header = f"{len(pdf_files)} PDFs merged on {timestamp}"
#     c.drawString(72, 750, header)
    
#     # Add separator line
#     c.line(72, 735, 540, 735)
    
#     # List of articles
#     c.setFont("Helvetica", 12)
#     y_position = 700
    
#     for i, pdf_path in enumerate(pdf_files, 1):
#         pdf_basename = os.path.basename(pdf_path).split('.')[0]
#         title = f"Article {i}: {pdf_basename}"
#         c.drawString(72, y_position, title)
#         y_position -= 20
        
#         if y_position < 50:
#             c.showPage()
#             c.setFont("Helvetica", 12)
#             y_position = 750
    
#     c.showPage()
#     c.save()
#     packet.seek(0)
#     first_pdf = PdfReader(packet)
#     writer.add_page(first_pdf.pages[0])

# def create_sample_pdf(filename, title, bookmarks, header_text):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
    
#     # Add header with source pdf basename
#     c.setFont("Helvetica", 10)
#     c.drawString(72, 800, header_text)
#     c.line(72, 795, 540, 795)
    
#     c.setFont("Helvetica", 12)
#     c.drawString(100, 750, title)
#     c.showPage()
#     c.save()
#     packet.seek(0)

#     new_pdf = PdfReader(packet)
#     writer = PdfWriter()
#     writer.add_page(new_pdf.pages[0])

#     for bookmark_title, page in bookmarks:
#         writer.add_outline_item(bookmark_title, page)

#     with open(filename, 'wb') as f:
#         writer.write(f)

# def add_separator_page(writer, pdf_basename, full_path, metadata):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
    
#     # Add clickable link
#     c.setFont("Helvetica-Bold", 16)
#     c.setFillColorRGB(0, 0, 1)  # Blue
    
#     # Create clickable link
#     text = pdf_basename
#     x, y = 100, 600
#     c.linkURL(full_path, (x, y-5, x+200, y+20), relative=1)
#     c.drawString(x, y, text)
    
#     # Add metadata
#     c.setFont("Helvetica", 12)
#     c.setFillColorRGB(0, 0, 0)  # Black
#     y_position = 500
#     for key, value in metadata.items():
#         c.drawString(100, y_position, f"{key}: {value}")
#         y_position -= 30
    
#     c.showPage()
#     c.save()
#     packet.seek(0)
#     separator_pdf = PdfReader(packet)
#     writer.add_page(separator_pdf.pages[0])

# def merge_pdfs_with_structure(pdf_files, output_path):
#     writer = PdfWriter()
    
#     # Sample metadata for each PDF (replace with actual metadata later)
#     sample_metadata = [
#         {
#             "Title": "First Research Paper",
#             "Author": "John Smith",
#             "Source": "Science Journal",
#             "Date": "2024-01-15"
#         },
#         {
#             "Title": "Second Research Paper",
#             "Author": "Jane Doe",
#             "Source": "Nature",
#             "Date": "2024-02-20"
#         },
#         {
#             "Title": "Third Research Paper",
#             "Author": "Bob Johnson",
#             "Source": "Research Quarterly",
#             "Date": "2024-03-10"
#         }
#     ]
    
#     # Add first page with timestamp and bookmark list
#     create_first_page(writer, pdf_files)
#     current_page = 1
    
#     for i, pdf_path in enumerate(pdf_files, 1):
#         pdf_basename = os.path.basename(pdf_path).split('.')[0]
#         full_path = os.path.abspath(pdf_path)
        
#         # Add separator page with bookmark and metadata
#         add_separator_page(writer, pdf_basename, full_path, sample_metadata[i-1])
#         article_title = f"Article {i}: {pdf_basename}"
#         separator_bookmark = writer.add_outline_item(article_title, current_page)
#         current_page += 1
        
#         # Add PDF content
#         pdf = PdfReader(pdf_path)
#         page_offset = current_page
        
#         for page in pdf.pages:
#             writer.add_page(page)
            
#         if pdf.outline:
#             seen_bookmarks = set()
#             for item in pdf.outline:
#                 if isinstance(item, dict) and '/Page' in item:
#                     title = item['/Title']
#                     if title not in seen_bookmarks:
#                         seen_bookmarks.add(title)
#                         page_num = pdf.get_destination_page_number(item)
#                         writer.add_outline_item(
#                             title,
#                             page_offset + page_num,
#                             parent=separator_bookmark
#                         )
        
#         current_page += len(pdf.pages)
    
#     with open(output_path, 'wb') as output:
#         writer.write(output)

# def verify_bookmarks(pdf_path):
#     reader = PdfReader(pdf_path)
    
#     def print_bookmark_tree(bookmarks, level=0):
#         for item in bookmarks:
#             if isinstance(item, list):
#                 print_bookmark_tree(item, level + 1)
#             else:
#                 indent = "  " * level
#                 page_num = reader.get_destination_page_number(item)
#                 print(f"{indent}- {item.title} (Page {page_num})")
    
#     print("\nBookmark structure:")
#     print_bookmark_tree(reader.outline)

# def run_tests():
#     print("Creating sample PDFs...")
    
#     # Create test PDFs with sections and headers
#     create_sample_pdf("article1.pdf", "Article 1", [
#         ("Section 1.1", 0),
#         ("Section 1.2", 0)
#     ], "article1")
    
#     create_sample_pdf("article2.pdf", "Article 2", [
#         ("Section 2.1", 0),
#         ("Section 2.2", 0)
#     ], "article2")
    
#     create_sample_pdf("article3.pdf", "Article 3", [
#         ("Section 3.1", 0),
#         ("Section 3.2", 0)
#     ], "article3")
    
#     print("Merging PDFs...")
#     pdf_files = ["article1.pdf", "article2.pdf", "article3.pdf"]
#     merge_pdfs_with_structure(pdf_files, "merged_articles.pdf")
    
#     print("Verifying merged PDF structure...")
#     verify_bookmarks("merged_articles.pdf")
    
#     print("\nTest completed. Please check merged_articles.pdf")

# if __name__ == "__main__":
#     run_tests()


In [12]:
# pretty good.  missing clickable links and basename headers

# from pypdf import PdfWriter, PdfReader
# from reportlab.pdfgen import canvas
# from reportlab.lib.pagesizes import letter
# from datetime import datetime
# import io
# import os

# def create_first_page(writer, pdf_files):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
    
#     # Add timestamp and PDF count header
#     timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
#     c.setFont("Helvetica-Bold", 14)
#     header = f"{len(pdf_files)} PDFs merged on {timestamp}"
#     c.drawString(72, 750, header)
    
#     # Add separator line
#     c.line(72, 735, 540, 735)
    
#     # List of articles
#     c.setFont("Helvetica", 12)
#     y_position = 700
    
#     for i, pdf_path in enumerate(pdf_files, 1):
#         pdf_basename = os.path.basename(pdf_path).split('.')[0]
#         title = f"Article {i}: {pdf_basename}"
#         c.drawString(72, y_position, title)
#         y_position -= 20
        
#         # Start new page if needed
#         if y_position < 50:
#             c.showPage()
#             c.setFont("Helvetica", 12)
#             y_position = 750
    
#     c.showPage()
#     c.save()
#     packet.seek(0)
#     first_pdf = PdfReader(packet)
#     writer.add_page(first_pdf.pages[0])

# def create_sample_pdf(filename, title, bookmarks):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
#     c.setFont("Helvetica", 12)
#     c.drawString(100, 750, title)
#     c.showPage()
#     c.save()
#     packet.seek(0)

#     new_pdf = PdfReader(packet)
#     writer = PdfWriter()
#     writer.add_page(new_pdf.pages[0])

#     for bookmark_title, page in bookmarks:
#         writer.add_outline_item(bookmark_title, page)

#     with open(filename, 'wb') as f:
#         writer.write(f)

# def add_separator_page(writer, pdf_basename):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
#     c.setFont("Helvetica-Bold", 16)
#     c.setFillColorRGB(0, 0, 1)  # Blue
#     c.drawString(100, 400, pdf_basename)
#     c.showPage()
#     c.save()
#     packet.seek(0)
#     separator_pdf = PdfReader(packet)
#     writer.add_page(separator_pdf.pages[0])

# def merge_pdfs_with_structure(pdf_files, output_path):
#     writer = PdfWriter()
    
#     # Add first page with timestamp and bookmark list
#     create_first_page(writer, pdf_files)
#     current_page = 1
    
#     for i, pdf_path in enumerate(pdf_files, 1):
#         pdf_basename = os.path.basename(pdf_path).split('.')[0]
        
#         # Add separator page with bookmark
#         add_separator_page(writer, pdf_basename)
#         article_title = f"Article {i}: {pdf_basename}"
#         separator_bookmark = writer.add_outline_item(article_title, current_page)
#         current_page += 1
        
#         # Add PDF content
#         pdf = PdfReader(pdf_path)
#         page_offset = current_page
        
#         # Add pages
#         for page in pdf.pages:
#             writer.add_page(page)
            
#         # Add original bookmarks under the separator
#         if pdf.outline:
#             seen_bookmarks = set()
#             for item in pdf.outline:
#                 if isinstance(item, dict) and '/Page' in item:
#                     title = item['/Title']
#                     if title not in seen_bookmarks:
#                         seen_bookmarks.add(title)
#                         page_num = pdf.get_destination_page_number(item)
#                         writer.add_outline_item(
#                             title,
#                             page_offset + page_num,
#                             parent=separator_bookmark
#                         )
        
#         current_page += len(pdf.pages)
    
#     with open(output_path, 'wb') as output:
#         writer.write(output)

# def verify_bookmarks(pdf_path):
#     reader = PdfReader(pdf_path)
    
#     def print_bookmark_tree(bookmarks, level=0):
#         for item in bookmarks:
#             if isinstance(item, list):
#                 print_bookmark_tree(item, level + 1)
#             else:
#                 indent = "  " * level
#                 page_num = reader.get_destination_page_number(item)
#                 print(f"{indent}- {item.title} (Page {page_num})")
    
#     print("\nBookmark structure:")
#     print_bookmark_tree(reader.outline)

# def run_tests():
#     print("Creating sample PDFs...")
    
#     # Create test PDFs with sections
#     create_sample_pdf("article1.pdf", "Article 1", [
#         ("Section 1.1", 0),
#         ("Section 1.2", 0)
#     ])
    
#     create_sample_pdf("article2.pdf", "Article 2", [
#         ("Section 2.1", 0),
#         ("Section 2.2", 0)
#     ])
    
#     create_sample_pdf("article3.pdf", "Article 3", [
#         ("Section 3.1", 0),
#         ("Section 3.2", 0)
#     ])
    
#     print("Merging PDFs...")
#     pdf_files = ["article1.pdf", "article2.pdf", "article3.pdf"]
#     merge_pdfs_with_structure(pdf_files, "merged_articles.pdf")
    
#     print("Verifying merged PDF structure...")
#     verify_bookmarks("merged_articles.pdf")
    
#     print("\nTest completed. Please check merged_articles.pdf")

# if __name__ == "__main__":
#     run_tests()


In [13]:
# TOC with separator pages, but dead links
# from pypdf import PdfWriter, PdfReader
# from reportlab.pdfgen import canvas
# from reportlab.lib.pagesizes import letter
# from reportlab.lib import colors
# import io
# import os

# def create_sample_pdf(filename, title, bookmarks):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
#     c.setFont("Helvetica", 12)
#     c.drawString(100, 750, title)
#     c.showPage()
#     c.save()
#     packet.seek(0)

#     new_pdf = PdfReader(packet)
#     writer = PdfWriter()
#     writer.add_page(new_pdf.pages[0])

#     for bookmark_title, page in bookmarks:
#         writer.add_outline_item(bookmark_title, page)

#     with open(filename, 'wb') as f:
#         writer.write(f)

# def create_toc_page(writer, pdf_files):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
#     c.setFont("Helvetica-Bold", 16)
#     c.setFillColorRGB(0, 0, 1)  # Blue
#     c.drawString(100, 750, "Table of Contents")
    
#     c.setFont("Helvetica", 12)
#     y_position = 700
#     for i, pdf_path in enumerate(pdf_files, 1):
#         pdf_basename = os.path.basename(pdf_path).split('.')[0]
#         c.drawString(100, y_position, f"Article {i}: {pdf_basename}")
#         y_position -= 30
    
#     c.showPage()
#     c.save()
#     packet.seek(0)
#     toc_pdf = PdfReader(packet)
#     writer.add_page(toc_pdf.pages[0])

# def add_separator_page(writer, pdf_basename):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
#     c.setFont("Helvetica-Bold", 16)
#     c.setFillColorRGB(0, 0, 1)  # Blue color
#     c.drawString(100, 400, pdf_basename)
#     c.showPage()
#     c.save()
#     packet.seek(0)
#     separator_pdf = PdfReader(packet)
#     writer.add_page(separator_pdf.pages[0])

# def merge_pdfs_with_structure(pdf_files, output_path):
#     writer = PdfWriter()
    
#     # Add Table of Contents
#     create_toc_page(writer, pdf_files)
#     toc_bookmark = writer.add_outline_item("Table of Contents", 0)
    
#     current_page = 1
    
#     for i, pdf_path in enumerate(pdf_files, 1):
#         pdf_basename = os.path.basename(pdf_path).split('.')[0]
        
#         # Add separator page with bookmark
#         add_separator_page(writer, pdf_basename)
#         article_title = f"Article {i}: {pdf_basename}"
#         separator_bookmark = writer.add_outline_item(article_title, current_page, parent=toc_bookmark)
#         current_page += 1
        
#         # Add PDF content
#         pdf = PdfReader(pdf_path)
#         page_offset = current_page
        
#         for page in pdf.pages:
#             writer.add_page(page)
        
#         # Add bookmarks from original PDF
#         if pdf.outline:
#             for item in pdf.outline:
#                 if isinstance(item, dict) and '/Page' in item:
#                     page_num = pdf.get_destination_page_number(item)
#                     writer.add_outline_item(
#                         item['/Title'],
#                         page_offset + page_num,
#                         parent=separator_bookmark
#                     )
        
#         current_page += len(pdf.pages)
    
#     with open(output_path, 'wb') as output:
#         writer.write(output)

# def verify_bookmarks(pdf_path):
#     reader = PdfReader(pdf_path)
    
#     def print_bookmark_tree(bookmarks, level=0):
#         for item in bookmarks:
#             if isinstance(item, list):
#                 print_bookmark_tree(item, level + 1)
#             else:
#                 indent = "  " * level
#                 page_num = reader.get_destination_page_number(item)
#                 print(f"{indent}- {item.title} (Page {page_num})")
    
#     print("\nBookmark structure:")
#     print_bookmark_tree(reader.outline)

# def run_tests():
#     print("Creating sample PDFs...")
    
#     create_sample_pdf("article1.pdf", "Article 1", [
#         ("Section 1.1", 0),
#         ("Section 1.2", 0)
#     ])
    
#     create_sample_pdf("article2.pdf", "Article 2", [
#         ("Section 2.1", 0),
#         ("Section 2.2", 0)
#     ])
    
#     create_sample_pdf("article3.pdf", "Article 3", [
#         ("Section 3.1", 0),
#         ("Section 3.2", 0)
#     ])
    
#     print("Merging PDFs...")
#     pdf_files = ["article1.pdf", "article2.pdf", "article3.pdf"]
#     merge_pdfs_with_structure(pdf_files, "merged_articles.pdf")
    
#     print("Verifying merged PDF structure...")
#     verify_bookmarks("merged_articles.pdf")
    
#     print("\nTest completed. Please check merged_articles.pdf")

# if __name__ == "__main__":
#     run_tests()


In [14]:
# works except TOC is blank
#
# from pypdf import PdfWriter, PdfReader
# from reportlab.pdfgen import canvas
# from reportlab.lib.pagesizes import letter
# from reportlab.lib import colors
# import io
# import os

# def create_sample_pdf(filename, title, bookmarks):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
#     c.setFont("Helvetica", 12)
#     c.drawString(100, 750, title)
#     c.showPage()
#     c.save()
#     packet.seek(0)

#     new_pdf = PdfReader(packet)
#     writer = PdfWriter()
#     writer.add_page(new_pdf.pages[0])

#     for bookmark_title, page in bookmarks:
#         writer.add_outline_item(bookmark_title, page)

#     with open(filename, 'wb') as f:
#         writer.write(f)

# def add_separator_page(writer, pdf_basename):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
#     c.setFont("Helvetica-Bold", 16)
#     c.setFillColorRGB(0, 0, 1)  # Blue color
#     c.drawString(100, 400, pdf_basename)
#     c.showPage()
#     c.save()
#     packet.seek(0)
#     separator_pdf = PdfReader(packet)
#     writer.add_page(separator_pdf.pages[0])

# def merge_pdfs_with_structure(pdf_files, output_path):
#     writer = PdfWriter()
    
#     # Add Table of Contents separator
#     add_separator_page(writer, "Table of Contents")
#     toc_bookmark = writer.add_outline_item("Table of Contents", 0)
    
#     current_page = 1
    
#     for i, pdf_path in enumerate(pdf_files, 1):
#         pdf_basename = os.path.basename(pdf_path).split('.')[0]
        
#         # Add separator page with bookmark
#         add_separator_page(writer, pdf_basename)
#         article_title = f"Article {i}: {pdf_basename}"
#         separator_bookmark = writer.add_outline_item(article_title, current_page)
#         current_page += 1
        
#         # Add PDF content
#         pdf = PdfReader(pdf_path)
#         page_offset = current_page
        
#         for page in pdf.pages:
#             writer.add_page(page)
        
#         # Add bookmarks from original PDF
#         if pdf.outline:
#             for item in pdf.outline:
#                 if isinstance(item, dict) and '/Page' in item:
#                     page_num = pdf.get_destination_page_number(item)
#                     writer.add_outline_item(
#                         item['/Title'],
#                         page_offset + page_num,
#                         parent=separator_bookmark
#                     )
        
#         current_page += len(pdf.pages)
    
#     with open(output_path, 'wb') as output:
#         writer.write(output)

# def verify_bookmarks(pdf_path):
#     reader = PdfReader(pdf_path)
    
#     def print_bookmark_tree(bookmarks, level=0):
#         for item in bookmarks:
#             if isinstance(item, list):
#                 print_bookmark_tree(item, level + 1)
#             else:
#                 indent = "  " * level
#                 page_num = reader.get_destination_page_number(item)
#                 print(f"{indent}- {item.title} (Page {page_num})")
    
#     print("\nBookmark structure:")
#     print_bookmark_tree(reader.outline)

# def run_tests():
#     print("Creating sample PDFs...")
    
#     # Create test PDFs with internal structure
#     create_sample_pdf("article1.pdf", "Article 1", [
#         ("Section 1.1", 0),
#         ("Section 1.2", 0)
#     ])
    
#     create_sample_pdf("article2.pdf", "Article 2", [
#         ("Section 2.1", 0),
#         ("Section 2.2", 0)
#     ])
    
#     create_sample_pdf("article3.pdf", "Article 3", [
#         ("Section 3.1", 0),
#         ("Section 3.2", 0)
#     ])
    
#     print("Merging PDFs...")
#     pdf_files = ["article1.pdf", "article2.pdf", "article3.pdf"]
#     merge_pdfs_with_structure(pdf_files, "merged_articles.pdf")
    
#     print("Verifying merged PDF structure...")
#     verify_bookmarks("merged_articles.pdf")
    
#     print("\nTest completed. Please check merged_articles.pdf")

# if __name__ == "__main__":
#     run_tests()


In [15]:
# was almost right.  repeated bookmarks @ end.
#
# from PyPDF2 import PdfMerger, PdfReader, PdfWriter
# from reportlab.pdfgen import canvas
# from reportlab.lib.pagesizes import letter
# import io
# import os

# def create_separator_page(title, number):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
#     c.setFont("Helvetica-Bold", 24)
#     c.drawCentredString(300, 400, f"Article {number}")
#     c.drawCentredString(300, 350, title)
#     c.save()
#     packet.seek(0)
#     return PdfReader(packet)

# def merge_pdfs_with_structure(pdf_files, output_path):
#     merger = PdfMerger()
#     current_page = 0
    
#     # Create and add TOC separator
#     toc_separator = create_separator_page("Table of Contents", 0)
#     merger.append(toc_separator)
#     merger.add_outline_item("Table of Contents", current_page)
#     current_page += 1
    
#     # Process each PDF file
#     for i, pdf_path in enumerate(pdf_files, 1):
#         title = os.path.splitext(os.path.basename(pdf_path))[0]
        
#         # Add separator page and its bookmark
#         separator = create_separator_page(title, i)
#         merger.append(separator)
        
#         # Create parent bookmark for this article
#         parent_bookmark = merger.add_outline_item(f"Article {i}: {title}", current_page)
#         current_page += 1
        
#         # Add the PDF content
#         with open(pdf_path, 'rb') as file:
#             pdf = PdfReader(file)
#             merger.append(file)
            
#             # Add article's internal bookmarks as children
#             outline = pdf.outline
#             if outline:
#                 for item in outline:
#                     if isinstance(item, dict) and '/Page' in item:
#                         page_num = pdf.get_destination_page_number(item)
#                         merger.add_outline_item(
#                             item['/Title'],
#                             current_page + page_num,
#                             parent=parent_bookmark
#                         )
            
#             current_page += len(pdf.pages)
    
#     with open(output_path, 'wb') as output:
#         merger.write(output)

# # Create test PDFs first
# def create_sample_pdf(filename, title, bookmarks):
#     packet = io.BytesIO()
#     c = canvas.Canvas(packet, pagesize=letter)
#     c.setFont("Helvetica", 12)
#     c.drawString(100, 750, title)
#     c.showPage()
#     c.save()
#     packet.seek(0)
    
#     new_pdf = PdfReader(packet)
#     writer = PdfWriter()
#     writer.add_page(new_pdf.pages[0])
    
#     for bookmark_title, page in bookmarks:
#         writer.add_outline_item(bookmark_title, page)
    
#     with open(filename, 'wb') as f:
#         writer.write(f)

# # Create test files
# create_sample_pdf("article1.pdf", "Article 1", [("Section 1.1", 0), ("Section 1.2", 0)])
# create_sample_pdf("article2.pdf", "Article 2", [("Section 2.1", 0), ("Section 2.2", 0)])
# create_sample_pdf("article3.pdf", "Article 3", [("Section 3.1", 0), ("Section 3.2", 0)])

# # Merge the PDFs
# pdf_files = ["article1.pdf", "article2.pdf", "article3.pdf"]
# merge_pdfs_with_structure(pdf_files, "merged_articles.pdf")


: 